# **Gradient Descent**
Vaguely based on: https://towardsdatascience.com/gradient-descent-from-scratch-e8b75fa986cc (dead link)

To start, we define a simple linear model:

In [1]:
import torch
import torch.nn as nn # nn is the neural network module from pytorch
class linFunc(nn.Module):
    def __init__(self):
        super().__init__()
        #the values below are often randomly initialized
        self.paramW = 0.0
        self.paramB = 0.0

    def forward(self, x):
        #function: (W * x) + b | where x is the input, W and b the parameters to be learned
        return self.paramW * x + self.paramB

**Loss Function:**

$MSE = \frac{1}{n} \sum_{i=1}^{n} (target_i - prediction_i)^2$

In [ ]:
def meanSquaredError(targets,predictions): #will only be used for demonstration here
    return torch.mean(targets-predictions)**2

**Linear Function with parameters $W$ and $b$ and input $x$:**

$f(x) = (W * x) + b$

**Therefore, we want to minimize the following function:**

$MSE = \frac{1}{n} \sum_{i=1}^{n} (target_i - (W * x_i) + b)^2$

**For this, we need the partial derivatives with respect to both $W$ and $b$:**
- $MSE = \frac{1}{n} \sum_{i=1}^{n} (target_i - ((W * x_i) + b))^2$

For composit functions, the chain rule applies:


---


1. Calculate the derivative of the outer Function (in this case: $x^2 => 2*x$)

- $Outer = \frac{1}{n} \sum_{i=1}^{n} 2*(target_i - ((W * x_i) + b))$


---


2. Calculate the partial derivatives of the inner function w.r.t params $b$ and $W$:

- $target - ((W*x)+b)$

  2.1 Assume everything to be constant (with the exception of $W$):  

  - $Inner_W(x) = const. - ((W*x) + const.)$

  - $Inner_W(x) = - W * x$

  - $Inner_W(x)' = -1 * x = -x$

  2.2 Assume everything to be constant (with the exception of $b$):
  
  - $Inner_b(x) = const. - ((const. * x) + b )$

  - $Inner_b(x) = - (x) + b = -x + b$

  - $Inner_b(x)' = -1$


---


3. Chain Rule: $Outer * Inner$

- $\frac{∂ MSE}{∂ W} = \frac{1}{n} \sum_{i=1}^{n} 2*(target_i - ((W * x_i) + b))*-x_i)$


- $\frac{∂ MSE}{∂ b} = \frac{1}{n} \sum_{i=1}^{n} 2*(target_i - ((W * x_i) + b))*-1)$

In [ ]:
def gradient(targets,ins,model): #normally this is done in parallel
    partialB = torch.mean(2*(targets-((ins*model.paramW)+model.paramB))*-1) #partial derivative for parameterized mse w.r.t. b
    partialW = torch.mean(2*(targets-((ins*model.paramW)+model.paramB))*-ins) #partial derivative for parameterized mse w.r.t. W
    return partialB,partialW

The gradient is a vector of the partial derivatives with respect to the model parameters:

$\nabla = \Biggr[\begin{matrix} \frac{∂ MSE}{∂ W} \\ \frac{∂ MSE}{∂ b} \end{matrix}\Biggr]$

The gradient is then used to update the parameters according to the current learning rate $lr$:

$params = params - (\nabla*lr)$

In [ ]:
def update(model,gradient,lr): #this is normally realized in a vectorized manner
    model.paramB = model.paramB - (gradient[0] * lr)
    model.paramW = model.paramW - (gradient[1] * lr)



---

**Application**

We want to find the following function:

$f(x) = (1*x) - 2$, so the parameters we are looking for are $W=1$ and $b=-2$

In [ ]:
samples = torch.FloatTensor([[[2],[0]],[[3],[1]],[[4],[2]],[[5],[3]]])
model = linFunc()
ins =  samples[:,0] #take the inputs from samples
tgts = samples[:,1] #take the targets from samples

for x in range(2500): #training for 2500 epochs
  predictions = model(ins) #prediction
  loss = meanSquaredError(tgts,predictions) #loss calculation
  gradients = gradient(tgts,ins,model) #gradient calculation
  update(model,gradients,0.01) #update with learning rate = 0.01

  '''if x % 100 == 0: #print out some info every 100 epochs
      print (f"\n~~~Iteration {x}~~~")
      print (f"Current loss: {loss}")
      print (f"PARAMS:\nb: {model.paramB}\nW: {model.paramW}")'''

print (f"\nFinal PARAMS:\nb: {model.paramB}\nW: {model.paramW}")
print (model(2))

# **Pytorch and Autograd**
Using PyTorch, the calculation of derivatives, gradient computation and parameter update are all automated

**nn.Parameter**

> Parameters are Tensor subclasses, that have a very special property when used with Module s - when they’re assigned as Module attributes they are automatically added to the list of its parameters, and will appear e.g. in parameters() iterator.

This is important because the parameters() iterator is passed to the optimizer!



**requires_grad()**

> Is True if gradients need to be computed for this Tensor, False otherwise.





**[Loss Functions](https://docs.pytorch.org/docs/stable/nn.html#loss-functions)**

- [torch.nn.MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)

- [torch.nn.CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

- [torch.nn.CosineEmbeddingLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CosineEmbeddingLoss.html#torch.nn.CosineEmbeddingLoss)

**[torch.optim](https://docs.pytorch.org/docs/stable/optim.html)**

Overall workflow:

    for input, target in dataset:
      1. optimizer.zero_grad()
      2. output = model(input)
      3. loss = loss_fn(output, target)
      4. loss.backward()
      5. optimizer.step()



<img src=https://pytorch.org/assets/images/augmented_computational_graph.png></img>

In [ ]:
import torch
import torch.nn as nn

###Simple Linear Model (2 parameters, 1 input, 1 output)
#compare __init__() from: https://pytorch.org/docs/stable/_modules/torch/nn/modules/linear.html#Linear
class linFunc(nn.Module):
    def __init__(self):
        super().__init__()
        #parameter W
        self.paramW = nn.Parameter(torch.zeros(1,requires_grad=True))
        #parameter b
        self.paramB = nn.Parameter(torch.zeros(1,requires_grad=True))

    def forward(self, x):
        #function: (W * x) + b | where x is the input, W and b the parameters to be learned
        return self.paramW * x + self.paramB

In [ ]:
samples = torch.FloatTensor([[[2],[0]],[[3],[1]],[[4],[2]],[[5],[3]]])
print (samples.shape)
print (samples.dim())

for s in samples:
    print (s)

'''
init model -> params are set according to __init__ function
defining the Loss function -> Mean Squared Error
defining the optimizer -> Stochastic Gradient Descent
lr = learning rate default 0.01 
'''

model = linFunc()
lossFct = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.01)

ins =  samples[:,0] #take the inputs from samples , basically for all rows, take the first column
tgts = samples[:,1] #take the targets from samples , basically for all rows, take the second column

for x in range(2500): #we train for 2500 epochs
  preds = model(ins) # prediction basically model forward pass
  loss = lossFct(preds,tgts) # loss calculation (compare predict with target)
  loss.backward() # compute gradients
  ###gradient is a tensor property
  #print (model.paramB.grad)
  ###gradient is a tensor property
  optimizer.step() #update params
  optimizer.zero_grad() #zero out gradients after each epoch

  if x % 100 == 0: #print out some info every 100 epochs
    print (f"\n~~~Iteration {x}~~~")
    print (f"Current loss: {float(loss)}")
    print (f"PARAMS:\nb: {float(model.paramB)}\nW: {float(model.paramW)}")
    
print (f"\nFinal PARAMS:\nb: {model.paramB}\nW: {model.paramW}")
print (model(2))

# **I Calculation exercise / Homework:**
Consider the following simple linear function with one trainable parameter $β$:

$f(x) = β + x$

The table below shows the expected (i.e. target) values for the input values in the left column:

inputs|targets
---|---
0|1
3|4
4|5
5|6

- Assume $β = 0$ and calculate the MSE

- The partial derivative of the MSE w.r.t $β$ given $f(x)$ is:

  - $\frac{1}{n} \sum_{i=1}^{n} - 2(target_i - (input_i + β))$

- Assume $β = 0$ and calculate the gradient based on the inputs and targets from the table

- Perform one parameter update of $β$, using a learning rate of 0.1 and the previously calculated gradient

In [ ]:
# solution below for excercise 1
import torch

inputs = torch.FloatTensor([[0],[3],[4],[5]])
targets = torch.FloatTensor([[1],[4],[5],[6]])

beta = torch.zeros(1,requires_grad=True)
model =  inputs + beta # Basic function y = x + b where b is beta

MSE = torch.mean((targets - model)**2)
print (f"Initial MSE: {MSE}")

# Gradient calculation
gradient = torch.mean(-2*(targets - (inputs + beta)))
print (f"Gradient: {gradient}")

# Update beta
learning_rate = 0.01
with torch.no_grad():
    beta -= learning_rate * gradient
print (f"Updated beta: {beta}")


# **II Programming Exercise:**

Use the model definition below and train the model **without using an optimizer**.
Instead, use the *.grad* property of the tensors and perform the parameter update yourself.

```
import torch
import torch.nn as nn

###Simple Linear Model (2 parameters, 1 input, 1 output)
#compare __init__() from: https://pytorch.org/docs/stable/_modules/torch/nn/modules/linear.html#Linear
class linFunc(nn.Module):
    def __init__(self):
        super().__init__()
        #parameter W
        self.paramW = nn.Parameter(torch.zeros(1,requires_grad=True))
        #parameter b
        self.paramB = nn.Parameter(torch.zeros(1,requires_grad=True))

    def forward(self, x):
        #function: (W * x) + b | where x is the input, W and b the parameters to be learned
        return self.paramW * x + self.paramB
```

Training loop (not fully implemented):



```
samples = torch.FloatTensor([[[2],[0]],[[3],[1]],[[4],[2]],[[5],[3]]])
model = linFunc()
lossFct = nn.MSELoss()
learningRate = 0.01

ins =  samples[:,0] #take the inputs from samples
tgts = samples[:,1] #take the targets from samples

for x in range(2500): #we train for 2500 epochs
  preds = model(ins) #prediction
  loss = lossFct(preds,tgts)

  ###Implement the gradient calculation and parameter update here!!!
```




In [ ]:
#  SOLUTION II Programming Exercise:

# Use the model definition below and train the model **without using an optimizer**.
# Instead, use the *.grad* property of the tensors and perform the parameter update yourself.

import torch
import torch.nn as nn

###Simple Linear Model (2 parameters, 1 input, 1 output)
#compare __init__() from: https://pytorch.org/docs/stable/_modules/torch/nn/modules/linear.html#Linear
class linFunc(nn.Module):
    def __init__(self):
        super().__init__()
        #parameter W
        self.paramW = nn.Parameter(torch.zeros(1,requires_grad=True))
        #parameter b
        self.paramB = nn.Parameter(torch.zeros(1,requires_grad=True))

    def forward(self, x):
        #function: (W * x) + b | where x is the input, W and b the parameters to be learned
        return self.paramW * x + self.paramB

In [ ]:
samples = torch.FloatTensor([[[2],[0]],[[3],[1]],[[4],[2]],[[5],[3]]])
model = linFunc()
lossFct = nn.MSELoss()
learningRate = 0.01

ins =  samples[:,0] #take the inputs from samples
tgts = samples[:,1] #take the targets from samples

for x in range(2500): #we train for 2500 epochs
  preds = model(ins) #prediction
  loss = lossFct(preds,tgts)

  ###Implement the gradient calculation and parameter update here!!!
  loss.backward() # compute gradients
  with torch.no_grad():
    model.paramW -= learningRate * model.paramW.grad
    model.paramB -= learningRate * model.paramB.grad
    model.paramW.grad.zero_()
    model.paramB.grad.zero_()

  if x % 100 == 0: #print out some info every 100 epochs
    print (f"\n~~~Iteration {x}~~~")
    print (f"Current loss: {float(loss)}")
    print (f"PARAMS:\nb: {float(model.paramB)}\nW: {float(model.paramW)}")